In [ ]:
Este notebook se usa para desarrollar paso a paso la aplicación de Streamlit que simula y visualiza predicciones para noviembre 2025 usando el modelo guardado y el dataset preparado.</VSCode.Cell>
<VSCode.Cell id="code1" language="python">import streamlit as st
import pandas as pd
import numpy as np
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

st.set_page_config(page_title="Simulación de Ventas Nov 2025", page_icon="🛍️", layout="wide")

MODEL_PATH = Path('models/modelo_final.joblib')
DATA_PATH = Path('data/processed/inferencia_df_transformado.csv')

model = None
df = None
error_message = None

try:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"No se encontró el modelo en {MODEL_PATH}")
    model = joblib.load(MODEL_PATH)
    if not hasattr(model, 'feature_names_in_'):
        raise AttributeError('El modelo no tiene feature_names_in_')
except Exception as e:
    error_message = f"Error cargando el modelo: {e}"

try:
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"No se encontró el archivo de inferencia en {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
except Exception as e:
    error_message = f"Error cargando los datos: {e}"

if error_message:
    st.error(error_message)
    st.stop()</VSCode.Cell>
<VSCode.Cell id="md2" language="markdown">## Configurar la página y el sidebar
Configuramos el sidebar con los controles de simulación: producto, descuento, escenario de competencia y botón de simulación.</VSCode.Cell>
<VSCode.Cell id="code2" language="python">with st.sidebar:
    st.markdown("## Controles de Simulación")
    productos = sorted(df['nombre'].unique())
    producto_seleccionado = st.selectbox('Producto', productos)
    descuento_pct = st.slider('Ajuste de descuento', -50, 50, 0, step=5, format='%d%%')
    escenario_competencia = st.radio(
        'Escenario de competencia',
        ['Actual (0%)', 'Competencia -5%', 'Competencia +5%']
    )
    ejecutar = st.button('Simular Ventas', type='primary')

st.markdown('---')

if not ejecutar:
    st.info('Selecciona un producto y ajustes, luego pulsa **Simular Ventas** para ver la proyección.')

if not ejecutar:
    st.stop()</VSCode.Cell>
<VSCode.Cell id="md3" language="markdown">## Filtrar producto y recalcular precios
Filtramos el dataframe para el producto seleccionado y recalculamos precio_venta, descuento_porcentaje y ratio_precio según el descuento elegido.</VSCode.Cell>
<VSCode.Cell id="code3" language="python">producto_df = df[df['nombre'] == producto_seleccionado].copy()
if producto_df.empty:
    st.error('No se encontró información para el producto seleccionado.')
    st.stop()

producto_df = producto_df.sort_values('fecha').reset_index(drop=True)
producto_df['precio_venta'] = producto_df['precio_base'] * (1 + descuento_pct / 100)
producto_df['descuento_porcentaje'] = descuento_pct
producto_df['ratio_precio'] = producto_df['precio_venta'] / producto_df['precio_competencia'].replace(0, np.nan)
producto_df['ratio_precio'] = producto_df['ratio_precio'].fillna(0)</VSCode.Cell>
<VSCode.Cell id="md4" language="markdown">## Ajustar competencia y recalcular variables
Aplicamos el ajuste de competencia a las columnas de Amazon, Decathlon y Deporvillage según el escenario seleccionado y actualizamos el ratio de precio.</VSCode.Cell>
<VSCode.Cell id="code4" language="python">competencia_factor = {
    'Actual (0%)': 0.0,
    'Competencia -5%': -0.05,
    'Competencia +5%': 0.05
}[escenario_competencia]

for col in ['Amazon', 'Decathlon', 'Deporvillage']:
    if col in producto_df.columns:
        producto_df[col] = producto_df[col] * (1 + competencia_factor)

# recalcular precio_competencia como promedio de los canales de competencia si existe
competencia_cols = [c for c in ['Amazon', 'Decathlon', 'Deporvillage'] if c in producto_df.columns]
if competencia_cols:
    producto_df['precio_competencia'] = producto_df[competencia_cols].mean(axis=1)
    producto_df['ratio_precio'] = producto_df['precio_venta'] / producto_df['precio_competencia'].replace(0, np.nan)
    producto_df['ratio_precio'] = producto_df['ratio_precio'].fillna(0)</VSCode.Cell>
<VSCode.Cell id="md5" language="markdown">## Ejecutar predicciones recursivas día a día
Ordenamos por fecha y hacemos predicciones uno a uno usando los lags del día 1 tal cual, luego actualizamos lag_1..lag_7 y rolling_mean_7 con las predicciones anteriores.</VSCode.Cell>
<VSCode.Cell id="code5" language="python">predicciones = []
producto_model_df = producto_df.copy().reset_index(drop=True)
feature_cols = list(model.feature_names_in_)
for idx in range(len(producto_model_df)):
    fila = producto_model_df.loc[idx].copy()
    valores_entrada = fila[feature_cols].to_numpy().reshape(1, -1)
    try:
        pred = model.predict(valores_entrada)[0]
    except Exception as e:
        st.error(f'Error en la predicción del día {idx + 1}: {e}')
        st.stop()
    predicciones.append(pred)

    if idx + 1 < len(producto_model_df):
        for lag_i in range(7, 1, -1):
            producto_model_df.at[idx + 1, f'lag_{lag_i}'] = producto_model_df.at[idx, f'lag_{lag_i-1}']
        producto_model_df.at[idx + 1, 'lag_1'] = pred
        ultimo_set = predicciones[-7:]
        producto_model_df.at[idx + 1, 'rolling_mean_7'] = np.mean(ultimo_set) if ultimo_set else producto_model_df.at[idx + 1, 'rolling_mean_7']

producto_df['unidades_predichas'] = np.round(predicciones).astype(int)
producto_df['ingresos_proyectados'] = producto_df['unidades_predichas'] * producto_df['precio_venta']
producto_df['precio_venta'] = producto_df['precio_venta'].round(2)
producto_df['precio_competencia'] = producto_df['precio_competencia'].round(2)</VSCode.Cell>
<VSCode.Cell id="md6" language="markdown">## Calcular KPIs y formatos numéricos
Calculamos los principales indicadores de la simulación y damos formato a los valores para mostrar en tarjetas y en la tabla.</VSCode.Cell>
<VSCode.Cell id="code6" language="python">unidades_totales = int(producto_df['unidades_predichas'].sum())
ingresos_totales = float(producto_df['ingresos_proyectados'].sum())
precio_promedio_venta = float(producto_df['precio_venta'].mean())
descuento_promedio = float(producto_df['descuento_porcentaje'].mean())

kpi_unidades = f"{unidades_totales:,d}"
kpi_ingresos = f"€{ingresos_totales:,.2f}"
kpi_precio = f"€{precio_promedio_venta:,.2f}"
kpi_descuento = f"{descuento_promedio:.0f}%"</VSCode.Cell>
<VSCode.Cell id="md7" language="markdown">## Graficar predicción diaria con Black Friday
Creamos un gráfico de línea con seaborn para las unidades predichas y marcamos el Black Friday con una línea vertical y una anotación clara.</VSCode.Cell>
<VSCode.Cell id="python7" language="python">fig, ax = plt.subplots(figsize=(12, 5))

sns.lineplot(
    x='dia_mes',
    y='unidades_predichas',
    data=producto_df,
    marker='o',
    color='#667eea',
    linewidth=2.5,
    ax=ax
)

black_friday = producto_df[producto_df['es_black_friday'] == 1]
if not black_friday.empty:
    dia_bf = int(black_friday['dia_mes'].iloc[0])
    unidades_bf = int(black_friday['unidades_predichas'].iloc[0])
    ax.axvline(dia_bf, color='#764ba2', linestyle='--', alpha=0.9)
    ax.scatter([dia_bf], [unidades_bf], color='red', s=100, zorder=5)
    ax.annotate(
        'Black Friday',
        xy=(dia_bf, unidades_bf),
        xytext=(dia_bf + 1, unidades_bf * 1.05),
        arrowprops=dict(color='red', arrowstyle='->'),
        color='red'
    )

ax.set_title(f'Predicción diaria de unidades vendidas - {producto_seleccionado}', fontsize=16, color='#333333')
ax.set_xlabel('Día de noviembre', fontsize=12)
ax.set_ylabel('Unidades predichas', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()</VSCode.Cell>
<VSCode.Cell id="md8" language="markdown">## Mostrar tabla detallada y comparativa de escenarios
Mostramos la tabla interactiva con los valores día a día y una sección comparativa de escenarios para unidades e ingresos.</VSCode.Cell>
<VSCode.Cell id="python8" language="python">st.title(f'📈 Simulación de Ventas - Noviembre 2025: {producto_seleccionado}')

col1, col2, col3, col4 = st.columns(4)
col1.metric('Unidades totales', kpi_unidades, delta=None)
col2.metric('Ingresos proyectados', kpi_ingresos, delta=None)
col3.metric('Precio promedio', kpi_precio, delta=None)
col4.metric('Descuento promedio', kpi_descuento, delta=None)

st.markdown('---')

st.pyplot(fig)

st.markdown('### Detalle día a día')
producto_df_table = producto_df[['fecha', 'dia_semana', 'precio_venta', 'precio_competencia', 'descuento_porcentaje', 'unidades_predichas', 'ingresos_proyectados']].copy()
producto_df_table['precio_venta'] = producto_df_table['precio_venta'].map('€{:.2f}'.format)
producto_df_table['precio_competencia'] = producto_df_table['precio_competencia'].map('€{:.2f}'.format)
producto_df_table['ingresos_proyectados'] = producto_df_table['ingresos_proyectados'].map('€{:.2f}'.format)
producto_df_table['descuento_porcentaje'] = producto_df_table['descuento_porcentaje'].map('{:.0f}%'.format)
producto_df_table['unidades_predichas'] = producto_df_table['unidades_predichas'].map('{:,.0f}'.format)

st.dataframe(producto_df_table.style.apply(
    lambda row: ['background-color: #ffe8e8' if row['fecha'] == '2025-11-28' else '' for _ in row], axis=1
).format({
    'precio_venta': '{}',
    'precio_competencia': '{}',
    'descuento_porcentaje': '{}',
    'unidades_predichas': '{}',
    'ingresos_proyectados': '{}'
}))

st.markdown('---')

st.markdown('### Comparativa de escenarios')

comparativa = []
for nombre, factor in [('Actual (0%)', 0.0), ('Competencia -5%', -0.05), ('Competencia +5%', 0.05)]:
    df_temp = producto_df.copy()
    for col in competencia_cols:
        df_temp[col] = df_temp[col] / (1 + competencia_factor) * (1 + factor)
    df_temp['precio_competencia'] = df_temp[competencia_cols].mean(axis=1)
    total_unidades = int(df_temp['unidades_predichas'].sum())
    total_ingresos = float((df_temp['unidades_predichas'] * df_temp['precio_venta']).sum())
    comparativa.append((nombre, total_unidades, total_ingresos))

c1, c2, c3 = st.columns(3)
for col, (nombre, unidades, ingresos) in zip((c1, c2, c3), comparativa):
    col.metric(f'{nombre}', f'{unidades:,d} unidades', f'€{ingresos:,.2f}')

st.markdown('---')

st.info('La simulación ha generado predicciones día a día usando el modelo guardado y lags actualizados recursivamente. Ajusta el descuento y el escenario para explorar impactos en ventas e ingresos.')
